# Super TikTakToe: the final model, on Google Colab

The **large** network continued from the winner of the tests, `sims_200` (iteration 279; 56.8% against `more_games` 260):

- 1024 games per iteration, all at once, **200 simulations** per move, every position learned from twice
- memory of 1.5 million positions (about 28 iterations)
- learning rate 0.0001, then 0.00003 for the last 15% of the time
- **36 hours of self-play and learning**, then it stops by itself (about 38 hours with the checks, about 75 units)
- every 8th iteration: a check against large 240 (512 games, 100 simulations) with the per-opening count
  worse-equal-better, and a snapshot
- every game is kept in the compact archive

**Everything goes into `Super-TikTakToe-models/final/`** (network, memory, log, snapshots, archive). The first start
copies the `sims_200` network and its memory and large's snapshot 240 there; nothing outside that folder is changed.

1. Menu **Runtime > Change runtime type**: choose the L4.
2. Run all cells from top to bottom. The first start takes a few minutes for the copies.
3. A Colab session ends after some hours: then simply run all cells again. The run continues where it stopped
   (at most the memory of the last 10 iterations is lost, the network never), and the time already used is counted.
4. Send me lines whenever you like. The last cell shows the checks.
5. When it has ended: **Runtime > Disconnect and delete runtime**.

In [ ]:
# Google Drive: the folder with the networks (experiments/sims_200 and snapshots/ of large)
from google.colab import drive
drive.mount('/content/drive')
MODELS = '/content/drive/MyDrive/Super-TikTakToe-models'
import os
os.environ['SUPER_TTT_MODELS'] = MODELS
assert (os.path.exists(f'{MODELS}/final/alphazero_large.pt')
        or os.path.exists(f'{MODELS}/experiments/sims_200/alphazero_large.pt')), 'sims_200 network missing'

In [ ]:
# The code from GitHub (the newest version every time)
%cd /content
!test -d Super-TikTakToe || git clone https://github.com/kasparhui/Super-TikTakToe.git
%cd /content/Super-TikTakToe
!git pull

In [ ]:
# Which graphics card did this session get? (it should be the L4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The final run. Every line is one iteration and shows the time used of the 36 hours.
# Stop with the stop button of this cell: everything is saved, and running it again continues.
!python -u src/train.py final

In [ ]:
# The checks against large 240 (512 games): points and openings worse / equal / better
import pandas as pd
path = f'{MODELS}/final/logs/alphazero_large_log.csv'
if not os.path.exists(path):
    path = f'{MODELS}/final/alphazero_large_log.csv'
log = pd.read_csv(path)
checks = log[log['vs_large240'].notna() & (log['iteration'] > 279)]
checks[['iteration', 'games', 'learning_rate', 'vs_large240', 'wins_large240', 'ties_large240', 'losses_large240',
        'worse_large240', 'equal_large240', 'better_large240', 'work_seconds', 'seconds']].tail(20)